# Western US Fire Season VPD Processing

Downloads gridMET vapor pressure deficit data for the western US (west of 100W)
during fire season (May-September) for 2010-2025.

**Scientific basis:** Abatzoglou, J.T. and Williams, A.P. (2016). Impact of
anthropogenic climate change on wildfire across western US forests. PNAS 113,
11770-11775. doi:10.1073/pnas.1607171113

**Data source:** gridMET (Abatzoglou 2013, DOI: 10.1002/joc.3413)
Accessed via OPeNDAP to avoid downloading full files (~500MB per year).

**Note:** VPD covers the western US only. National acres burned on the main chart
include the entire country. These series show related climate context, not a
direct national correlation.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
YEARS = list(range(1979, 2026))
LON_MIN, LON_MAX = -125, -100  # western US boundary per Abatzoglou & Williams 2016
LAT_MIN, LAT_MAX = 25, 50
BASE_URL = "http://thredds.northwestknowledge.net/thredds/dodsC/MET/vpd/vpd_{year}.nc"
OUTPUT_PATH = "../data/vpd-annual.csv"
VPD_VAR = "mean_vapor_pressure_deficit"  # gridMET long name; standard_name is vpd

In [ ]:
def fetch_seasonal_vpd(year):
    url = BASE_URL.format(year=year)
    ds = xr.open_dataset(url)
    vpd = ds[VPD_VAR]
    # OPeNDAP returns only the spatial and temporal subset, not the full ~500MB file
    # gridMET latitude runs north to south; slice high to low
    subset = vpd.sel(
        lon=slice(LON_MIN, LON_MAX),
        lat=slice(LAT_MAX, LAT_MIN),
        day=slice(f"{year}-05-01", f"{year}-09-30"),
    )
    mean_vpd = float(subset.mean(skipna=True).values)
    ds.close()
    return round(mean_vpd, 3)

In [ ]:
results = []
for year in YEARS:
    try:
        vpd = fetch_seasonal_vpd(year)
        results.append({"year": year, "vpd_kpa": vpd})
        print(f"{year}: {vpd} kPa")
    except Exception as e:
        print(f"{year}: FAILED: {e}")
        results.append({"year": year, "vpd_kpa": None})

failed = [r["year"] for r in results if r["vpd_kpa"] is None]
if failed:
    print(f"WARNING: missing years: {failed}")

In [ ]:
df = pd.DataFrame(results)
print(df.to_string(index=False))
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

In [ ]:
df_clean = df.dropna()
plt.figure(figsize=(10, 4))
plt.plot(df_clean["year"], df_clean["vpd_kpa"], marker="o", color="#7b5ea7", linewidth=2)
plt.title("Western US Fire Season VPD 2010-2025 (gridMET)")
plt.xlabel("Year")
plt.ylabel("VPD (kPa)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Source Citation

Abatzoglou, J.T. (2013). Development of daily high-resolution climate surfaces
for the contiguous United States of America. International Journal of Climatology.
DOI: 10.1002/joc.3413

Spatial extent: Western US (west of 100W longitude)
Temporal subset: May-September fire season
Years processed: 2010-2025
Data accessed via OPeNDAP: http://thredds.northwestknowledge.net/thredds/dodsC/MET/vpd/